# Small factor groups across frozen SAE seeds

Do a few existing factors recover contribution agreement missed by singletons?
Groups retain their family anchors. Selection uses reference discovery patients;
evaluation uses an independent frozen patient role. Results describe these frozen
dictionaries, not clinical concepts or functional interchangeability.


In [1]:
import itertools
import os
import pickle
import time
from functools import cmp_to_key
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from cmr_common import setup_run, iter_units, freeze_write, freeze_read, publish, patient_weights, finish_run

MODE = os.environ.get("CMR_MODE", "check")
SEEDS = (42, 135, 215)
PAIRS = tuple(itertools.combinations(SEEDS, 2))
SIZES = tuple(itertools.product(range(1, 6), repeat=2))
SHORTLIST, BEAM, CONTROLS = 20, 2, 3
TIE, EPSILON = 1e-6, 1e-12
INTERVAL_SEEDS = tuple(range(3001, 3501))
BUDGET_SECONDS = 20 * 60 if MODE == "smoke" else 8 * 3600
ctx = setup_run("groups")
frozen, group_rows, curve_rows, geometry_rows, support_rows = {}, [], [], [], []
print(f"Mode: {MODE}; three seed pairs; anchored groups of one to five; no outcomes used.")

Mode: evaluate; three seed pairs; anchored groups of one to five; no outcomes used.


## Contribution moments

Sum each factor's actual activation times its decoder direction. Raw directions
include the reference scale, with no fitted weights, rotations or biases. Small
inner-product matrices score candidate groups exactly without patient embeddings.


In [2]:
def contribution_moments(left_a, right_a, left_d, right_d, device):
    a, b, d, e = [torch.as_tensor(x, dtype=torch.float64, device=device)
                  for x in (left_a, right_a, left_d, right_d)]
    with torch.inference_mode():
        matrices = ((a.T @ a) * (d @ d.T), (b.T @ b) * (e @ e.T),
                    (a.T @ b) * (d @ e.T))
    return tuple(x.cpu().numpy() for x in matrices)

def moment_error(moments, left, right):
    ml, mr, cross = moments
    l = float(ml[np.ix_(left, left)].sum())
    r = float(mr[np.ix_(right, right)].sum())
    denominator = l + r
    if not np.isfinite(denominator):
        raise ValueError("Nonfinite contribution moments")
    if denominator <= EPSILON:
        return None
    numerator = denominator - 2 * float(cross[np.ix_(left, right)].sum())
    if numerator < -1e-8 * denominator:
        raise ValueError("Negative squared contribution disagreement")
    return max(0., numerator) / denominator

def actual_contribution(activation, decoder, factors):
    a = torch.as_tensor(activation[:, list(factors)], dtype=torch.float64, device=ctx["device"])
    d = torch.as_tensor(decoder[list(factors)], dtype=torch.float64, device=ctx["device"])
    return (a @ d).cpu().numpy()

def alive_factors(activation):
    return np.flatnonzero((activation > 0).any(axis=0) & (np.ptp(activation, axis=0) > 0))

if MODE != "check":
    print("Candidate moments use only the fitting/tuning rows exposed by the read-only adapter.")

Candidate moments use only the fitting/tuning rows exposed by the read-only adapter.


## Bounded fitting search

Each side first keeps at most 20 fitting-ranked additions. Incremental search
retains two alternatives at each of the 25 size pairs, including temporarily worse
intermediates. Numerical ties use sorted factor IDs. This is a bounded search,
not a guarantee of the globally best group.


In [3]:
def ordered_candidates(candidates, score, tuning=False):
    def compare(a, b):
        x, y = score(a), score(b)
        x, y = np.inf if x is None else x, np.inf if y is None else y
        if x < y - TIE:
            return -1
        if y < x - TIE:
            return 1
        ka = (sum(map(len, a)), max(map(len, a)), a) if tuning else a
        kb = (sum(map(len, b)), max(map(len, b)), b) if tuning else b
        return (ka > kb) - (ka < kb)
    return sorted(set(candidates), key=cmp_to_key(compare))

def addition_shortlists(moments, anchors, alive, excluded):
    left, right = (anchors[0],), (anchors[1],)
    output = []
    for side in (0, 1):
        candidates = []
        for factor in alive[side]:
            if factor in excluded[side] or factor == anchors[side]:
                continue
            pair = (tuple(sorted((*left, int(factor)))), right) if side == 0 else (left, tuple(sorted((*right, int(factor)))))
            candidates.append(pair)
        ranked = ordered_candidates(candidates, lambda pair: moment_error(moments, *pair))
        output.append(tuple(next(f for f in pair[side] if f != anchors[side]) for pair in ranked[:SHORTLIST]))
    return tuple(output)

def bounded_search(moments, anchors, alive, excluded):
    shortlists = addition_shortlists(moments, anchors, alive, excluded)
    retained = {(1, 1): [((int(anchors[0]),), (int(anchors[1]),))]}
    for k, l in sorted(SIZES, key=lambda size: (sum(size), size)):
        if (k, l) == (1, 1):
            continue
        candidates = []
        for side, predecessor in ((0, (k - 1, l)), (1, (k, l - 1))):
            for pair in retained.get(predecessor, []):
                for factor in shortlists[side]:
                    if factor not in pair[side]:
                        groups = list(pair)
                        groups[side] = tuple(sorted((*groups[side], factor)))
                        candidates.append(tuple(groups))
        retained[k, l] = ordered_candidates(candidates, lambda pair: moment_error(moments, *pair))[:BEAM]
    return {"anchors": tuple(map(int, anchors)), "shortlists": shortlists, "candidates": retained,
            "fitting_ledger": [{"size": size, "groups": pair, "fit_E": moment_error(moments, *pair)}
                               for size, pairs in retained.items() for pair in pairs]}

if MODE != "check":
    print("Search bounds: 20 additions per side, two retained candidates per size, 25 size pairs.")

Search bounds: 20 additions per side, two retained candidates per size, 25 size pairs.


## Matching controls and freezing tuning choices

Three non-family anchor pairs match fitting prevalence and raw contribution RMS
in empirical percentile coordinates. Each gets the same search capacity. Tuning
selects one candidate per size and then the overall size; evaluation never reselects.


In [4]:
def empirical_ranks(values):
    values = np.asarray(values)
    order = np.argsort(values, kind="stable")
    ranks = np.empty(len(values), dtype=float)
    for value in np.unique(values):
        positions = np.flatnonzero(values[order] == value)
        ranks[order[positions]] = (positions.mean() + .5) / len(values)
    return ranks

def matched_anchors(activations, decoders, anchors, excluded):
    matched = []
    for a, d, anchor, blocked in zip(activations, decoders, anchors, excluded):
        rms = np.sqrt((a * a).mean(axis=0) * (d * d).sum(axis=1))
        ranks = np.column_stack([empirical_ranks((a > 0).mean(axis=0)), empirical_ranks(rms)])
        distance = ((ranks - ranks[anchor]) ** 2).sum(axis=1)
        available = [int(j) for j in alive_factors(a) if j not in blocked]
        matched.append(sorted(available, key=lambda j: (distance[j], j))[:CONTROLS])
    return [tuple(side[i] for side in matched) for i in range(min(map(len, matched)))]

def tune_search(search, moments):
    winners, ledger = {}, []
    for size, candidates in search["candidates"].items():
        ranked = ordered_candidates(candidates, lambda pair: moment_error(moments, *pair), tuning=True)
        for candidate in ranked:
            ledger.append({"size": size, "groups": candidate, "tune_E": moment_error(moments, *candidate)})
        if ranked and moment_error(moments, *ranked[0]) is not None:
            winners[size] = ranked[0]
    ranked = ordered_candidates(winners.values(), lambda pair: moment_error(moments, *pair), tuning=True)
    return {**search, "size_winners": winners, "selected": ranked[0] if ranked else None, "ledger": ledger}

def develop_pair(unit, seeds, device):
    fit, tune = unit["fit"], unit["tune"]
    a = [unit["codes"][s] for s in seeds]
    d = [unit["decoders"][s] * unit["scale"] for s in seeds]
    anchors = tuple(unit["members"][s] for s in seeds)
    excluded = tuple({anchor} for anchor in anchors)
    fitting = contribution_moments(a[0][fit], a[1][fit], *d, device)
    tuning = contribution_moments(a[0][tune], a[1][tune], *d, device)
    alive = tuple(alive_factors(values[fit]) for values in a)
    controls = matched_anchors([values[fit] for values in a], d, anchors, excluded)
    searched = [tune_search(bounded_search(fitting, pair, alive, excluded), tuning)
                for pair in [anchors, *controls]]
    return {"seeds": seeds, "true": searched[0], "controls": searched[1:],
            "control_count": len(controls), "identity": unit["identity"]}

if MODE != "check":
    print("Three control attempts per pair; unavailable matches remain explicit.")

Three control attempts per pair; unavailable matches remain explicit.


## Reference universe and independent freeze

Use only the saved eligible correspondence families in last_3 and all_history,
2007–2015. Effective aliases share computation. The primary period is 2007–2013;
2014–2015 results concern representation only. No mortality input is used.


In [5]:
if MODE != "check":
    if MODE == "evaluate":
        frozen = freeze_read(ctx)
    for unit in iter_units(ctx, "groups"):
        if time.monotonic() - ctx["started"] >= BUDGET_SECONDS:
            finish_run(ctx, "budget_incomplete", {"completed_pairs": len(frozen)})
            raise TimeoutError("Factor-group budget exhausted; completed units retained privately")
        if not unit.get("identity"):
            raise ValueError("Missing frozen group input identity")
        for seeds in PAIRS:
            pair_started = time.monotonic()
            key = f'{unit["key"]}/{seeds[0]}-{seeds[1]}'
            if MODE == "evaluate":
                if key not in frozen or frozen[key]["identity"] != unit["identity"]:
                    raise ValueError("Frozen group identity or universe mismatch")
            else:
                frozen[key] = develop_pair(unit, seeds, ctx["device"])
            entry = frozen[key]
            chosen = entry["true"]["selected"]
            support_rows.append({"concept": str(unit["family"]), "reference": int(unit["reference"]),
                "pair": f"{seeds[0]}-{seeds[1]}", "aliases": ",".join(unit["aliases"]),
                "fit_rows": len(unit["fit"]), "tune_rows": len(unit["tune"]),
                "evaluation_rows": len(unit["evaluation"]), "controls": entry["control_count"],
                "seconds": time.monotonic() - pair_started,
                "status": "measured" if chosen else "undefined"})
        if MODE in {"smoke", "develop"}:
            with (ctx["out"] / "private" / "groups-progress.pkl").open("wb") as handle:
                pickle.dump(frozen, handle)
    if MODE == "develop":
        print("Frozen branch:", freeze_write(ctx, frozen))
    display(pd.DataFrame(support_rows))

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,concept,reference,pair,aliases,fit_rows,tune_rows,evaluation_rows,controls,seconds,status
0,"2007/42/42:94,135:3,215:146",2007,42-135,"last_3,all_history",0,0,237,3,0.000011,measured
1,"2007/42/42:94,135:3,215:146",2007,42-215,"last_3,all_history",0,0,237,3,0.000004,measured
2,"2007/42/42:94,135:3,215:146",2007,135-215,"last_3,all_history",0,0,237,3,0.000004,measured
3,"2007/42/42:65,135:187,215:78",2007,42-135,"last_3,all_history",0,0,237,3,0.000008,measured
4,"2007/42/42:65,135:187,215:78",2007,42-215,"last_3,all_history",0,0,237,3,0.000004,measured
...,...,...,...,...,...,...,...,...,...,...
331,"2014/42/42:284,135:28,215:279",2014,42-215,all_history,0,0,325,3,0.000003,measured
332,"2014/42/42:284,135:28,215:279",2014,135-215,all_history,0,0,325,3,0.000003,measured
333,"2014/42/42:73,135:222,215:96",2014,42-135,all_history,0,0,325,3,0.000006,measured
334,"2014/42/42:73,135:222,215:96",2014,42-215,all_history,0,0,325,3,0.000003,measured


## Conditional patient uncertainty and contribution diagnostics

Evaluate the frozen groups once on reference evaluation patients. Compare each
with its freshly computed singleton and equally searched size-matched controls.
Shared patient-cluster draws hold fitted groups fixed. Empty strata and zero
contribution energy remain undefined.


In [6]:
def row_energies(left, right):
    return ((left - right) ** 2).sum(axis=1), (left * left).sum(axis=1) + (right * right).sum(axis=1)

def weighted_errors(numerator, denominator, weights):
    w = torch.as_tensor(weights, dtype=torch.float64, device=ctx["device"])
    values = torch.as_tensor(np.column_stack([numerator, denominator]), dtype=torch.float64, device=ctx["device"])
    n, d = (w @ values).cpu().numpy().T
    return np.divide(n, d, out=np.full(np.shape(d), np.nan), where=d > EPSILON)

def interval(values):
    finite = np.asarray(values)[np.isfinite(values)]
    return (float(np.quantile(finite, .025)), float(np.quantile(finite, .975)),
            1 - len(finite) / len(values)) if len(finite) else (None, None, 1.)

def contribution_diagnostics(left, right, weights):
    numerator, denominator = row_energies(left, right)
    error = float(numerator.sum() / denominator.sum()) if denominator.sum() > EPSILON else None
    ln, rn = np.linalg.norm(left, axis=1), np.linalg.norm(right, axis=1)
    usable = (ln > EPSILON) & (rn > EPSILON)
    cosine = (left[usable] * right[usable]).sum(axis=1) / (ln[usable] * rn[usable])
    lo, hi, undefined = interval(weighted_errors(numerator, denominator, weights))
    return {"E": error, "E_low": lo, "E_high": hi, "undefined_draw_fraction": undefined,
            "left_rms": float(np.sqrt((left * left).sum(axis=1).mean())) if len(left) else None,
            "right_rms": float(np.sqrt((right * right).sum(axis=1).mean())) if len(right) else None,
            "zero_norm_fraction": float((~usable).mean()) if len(usable) else None,
            "mean_signed_cosine": float(cosine.mean()) if len(cosine) else None}

def basis(decoder):
    if not len(decoder):
        return np.zeros((decoder.shape[1], 0))
    _, singular, vh = np.linalg.svd(decoder, full_matrices=False)
    tolerance = max(decoder.shape) * np.finfo(float).eps * singular[0] if len(singular) else 0.
    return vh[singular > tolerance].T

def group_geometry(left_d, right_d, left_c, right_c):
    ql, qr = basis(left_d), basis(right_d)
    spectrum = np.linalg.svd(ql.T @ qr, compute_uv=False) ** 2
    el, er = float((left_c * left_c).sum()), float((right_c * right_c).sum())
    return {"left_rank": ql.shape[1], "right_rank": qr.shape[1], "spectrum": spectrum,
            "left_projection_energy": float(((left_c @ qr) ** 2).sum() / el) if el > EPSILON else None,
            "right_projection_energy": float(((right_c @ ql) ** 2).sum() / er) if er > EPSILON else None}

if MODE != "check":
    print("Intervals use 500 shared patient draws; no group search inside draws.")

Intervals use 500 shared patient draws; no group search inside draws.


## Frozen scoring, size curves and anchor specificity

The original medoid's A stratum is identical for all three seed pairs. Report raw
and standardized geometry with actual ranks. Dropping each anchor and showing
energy inside/outside A tests specificity; these diagnostics cannot reselect groups.


Compare frozen full/high-stratum errors with singleton and selected groups using the same patient draws.

In [7]:
def scoped_scores(common, contributions, baseline, selected, weights, high, groups, patients):
    output = []
    for stratum, mask in (("full", np.ones(len(high), dtype=bool)), ("original_high", high)):
        l, r = contributions[0][mask], contributions[1][mask]
        result = contribution_diagnostics(l, r, weights[:, mask])
        n, d = row_energies(l, r)
        draw = weighted_errors(n, d, weights[:, mask])
        bn, bd = row_energies(baseline[0][mask], baseline[1][mask])
        sn, sd = row_energies(selected[0][mask], selected[1][mask])
        base_E = bn.sum()/bd.sum() if bd.sum() > EPSILON else None
        selected_E = sn.sum()/sd.sum() if sd.sum() > EPSILON else None
        blo, bhi, bu = interval(draw - weighted_errors(bn, bd, weights[:, mask]))
        slo, shi, su = interval(draw - weighted_errors(sn, sd, weights[:, mask]))
        output.append({**common, **result, "stratum": stratum, "rows": int(mask.sum()),
            "distinct_count": len(np.unique(patients[mask])), "status": "measured" if result["E"] is not None else "undefined",
            "left_factors": ",".join(map(str, groups[0])), "right_factors": ",".join(map(str, groups[1])),
            "left_size": len(groups[0]), "right_size": len(groups[1]),
            "delta_singleton": result["E"]-base_E if result["E"] is not None and base_E is not None else None,
            "delta_low": blo, "delta_high": bhi, "delta_undefined_fraction": bu,
            "delta_selected": result["E"]-selected_E if result["E"] is not None and selected_E is not None else None,
            "delta_selected_low": slo, "delta_selected_high": shi, "delta_selected_undefined_fraction": su})
    return output

if MODE != "check":
    print('Compare frozen full/high-stratum errors with singleton and selected groups using the same patient draws.')


Compare frozen full/high-stratum errors with singleton and selected groups using the same patient draws.


Report both coordinate systems, the principal-angle spectrum, ranks and directed energy fractions.

In [8]:
def geometry_records(common, activations, decoders, scale, groups):
    output = []
    for coordinate in ("raw", "standardized"):
        d = [decoder * scale for decoder in decoders] if coordinate == "raw" else decoders
        c = [actual_contribution(a, decoder, factors) for a, decoder, factors in zip(activations, d, groups)]
        geometry = group_geometry(d[0][list(groups[0])], d[1][list(groups[1])], *c)
        for angle in range(max(1, len(geometry["spectrum"]))):
            output.append({**common, "coordinate": coordinate, "angle": angle + 1,
                "squared_cosine": float(geometry["spectrum"][angle]) if len(geometry["spectrum"]) else None,
                **{key: value for key, value in geometry.items() if key != "spectrum"}})
    return output

if MODE != "check":
    print('Report both coordinate systems, the principal-angle spectrum, ranks and directed energy fractions.')


Report both coordinate systems, the principal-angle spectrum, ranks and directed energy fractions.


Drop each candidate's own anchors and quantify contribution energy inside and outside original A.

In [9]:
def specificity_records(common, activations, decoders, groups, anchors, high):
    output = []
    for side in (0, 1):
        dropped = list(groups)
        dropped[side] = tuple(factor for factor in groups[side] if factor != anchors[side])
        removed = [actual_contribution(a, decoder, factors) for a, decoder, factors in zip(activations, decoders, dropped)]
        numerator, denominator = row_energies(*removed)
        output.append({**common, "stratum": f"drop_anchor_{side}",
            "status": "measured" if denominator.sum() > EPSILON else "undefined",
            "E": float(numerator.sum()/denominator.sum()) if denominator.sum() > EPSILON else None})
        contribution = actual_contribution(activations[side], decoders[side], groups[side])
        energy = (contribution * contribution).sum(axis=1)
        output.append({**common, "stratum": f"energy_side_{side}", "status": "measured",
            "energy_inside_A": float(energy[high].sum()), "energy_outside_A": float(energy[~high].sum())})
    return output

if MODE != "check":
    print("Drop each candidate's own anchors and quantify contribution energy inside and outside original A.")


Drop each candidate's own anchors and quantify contribution energy inside and outside original A.


Each true selection has one fresh singleton and three equally searched, size-matched control attempts.

In [10]:
def evaluation_candidates(entry):
    chosen, anchors = entry["true"]["selected"], entry["true"]["anchors"]
    candidates = [("selected", chosen, anchors), ("singleton", tuple((anchor,) for anchor in anchors), anchors)]
    size = tuple(map(len, chosen))
    for index in range(CONTROLS):
        control = entry["controls"][index] if index < len(entry["controls"]) else None
        candidates.append((f"control_{index+1}", None if control is None else control["size_winners"].get(size),
                           None if control is None else control["anchors"]))
    return candidates

if MODE != "check":
    print('Each true selection has one fresh singleton and three equally searched, size-matched control attempts.')


Each true selection has one fresh singleton and three equally searched, size-matched control attempts.


Keep failed group choices in every denominator and still recompute the original singleton on evaluation rows.

In [11]:
def unavailable_pair(unit, entry, weights):
    rows, seeds = unit["evaluation"], entry["seeds"]
    high = unit["activation"][rows] > unit["cutoff"]
    singleton = [actual_contribution(unit["codes"][seed][rows], unit["decoders"][seed] * unit["scale"],
                                    (unit["members"][seed],)) for seed in seeds]
    output = []
    for kind in ("selected", "singleton", "control_1", "control_2", "control_3"):
        for stratum, mask in (("full", np.ones(len(rows), dtype=bool)), ("original_high", high)):
            diagnostic = contribution_diagnostics(singleton[0][mask], singleton[1][mask], weights[:, mask]) if kind == "singleton" else {"E": None}
            output.append({"kind": kind, "pair": f"{seeds[0]}-{seeds[1]}", "concept": str(unit["family"]),
                "reference": int(unit["reference"]), "period": "primary" if unit["reference"] <= 2013 else "secondary",
                "aliases": ",".join(unit["aliases"]), "stratum": stratum, "rows": int(mask.sum()),
                "distinct_count": len(np.unique(unit["patients"][rows][mask])), **diagnostic,
                "delta_singleton": None, "delta_selected": None,
                "status": "measured" if diagnostic["E"] is not None else "undefined"})
    return output, [], []

if MODE != "check":
    print('Keep failed group choices in every denominator and still recompute the original singleton on evaluation rows.')


Keep failed group choices in every denominator and still recompute the original singleton on evaluation rows.


Score each frozen seed pair without fitting or choosing new factors.

In [12]:
def evaluate_pair(unit, entry, weights):
    seeds, chosen = entry["seeds"], entry["true"]["selected"]
    if chosen is None:
        return unavailable_pair(unit, entry, weights)
    rows = unit["evaluation"]
    a, d = [unit["codes"][s][rows] for s in seeds], [unit["decoders"][s] for s in seeds]
    raw = [decoder * unit["scale"] for decoder in d]
    high = unit["activation"][rows] > unit["cutoff"]
    candidates = evaluation_candidates(entry)
    baseline = [actual_contribution(x, dec, factors) for x, dec, factors in zip(a, raw, candidates[1][1])]
    selected = [actual_contribution(x, dec, factors) for x, dec, factors in zip(a, raw, chosen)]
    output, geometry, curves = [], [], []
    for label, groups, anchors in candidates:
        common = {"kind": label, "pair": f"{seeds[0]}-{seeds[1]}", "concept": str(unit["family"]),
            "reference": int(unit["reference"]), "period": "primary" if unit["reference"] <= 2013 else "secondary",
            "aliases": ",".join(unit["aliases"])}
        if groups is None:
            for stratum in ("full", "original_high"):
                output.append({**common, "stratum": stratum, "status": "unsupported", "E": None,
                               "delta_singleton": None, "delta_selected": None})
            continue
        contribution = [actual_contribution(x, dec, factors) for x, dec, factors in zip(a, raw, groups)]
        output.extend(scoped_scores(common, contribution, baseline, selected, weights, high, groups, unit["patients"][rows]))
        output.extend(specificity_records(common, a, raw, groups, anchors, high))
        geometry.extend(geometry_records(common, a, d, unit["scale"], groups))
    for size, groups in entry["true"]["size_winners"].items():
        values = [actual_contribution(x, dec, factors) for x, dec, factors in zip(a, raw, groups)]
        numerator, denominator = row_energies(*values)
        curves.append({"concept": str(unit["family"]), "reference": int(unit["reference"]),
            "pair": f"{seeds[0]}-{seeds[1]}", "aliases": ",".join(unit["aliases"]),
            "period": "primary" if unit["reference"] <= 2013 else "secondary",
            "left_size": size[0], "right_size": size[1],
            "E": float(numerator.sum()/denominator.sum()) if denominator.sum() > EPSILON else None,
            "selected": groups == chosen})
    return output, curves, geometry

if MODE != "check":
    print('Score each frozen seed pair without fitting or choosing new factors.')


Score each frozen seed pair without fitting or choosing new factors.


In [13]:
if MODE == "evaluate":
    for unit in iter_units(ctx, "groups"):
        if time.monotonic() - ctx["started"] >= BUDGET_SECONDS:
            finish_run(ctx, "budget_incomplete", {"completed_pairs": len(group_rows)})
            raise TimeoutError("Factor-group evaluation budget exhausted")
        rows = unit["evaluation"]
        weights = patient_weights(unit["patients"][rows], seeds=INTERVAL_SEEDS)
        for seeds in PAIRS:
            key = f'{unit["key"]}/{seeds[0]}-{seeds[1]}'
            result, curves, geometry = evaluate_pair(unit, frozen[key], weights)
            group_rows.extend(result); curve_rows.extend(curves); geometry_rows.extend(geometry)
    display(pd.DataFrame(group_rows))
    display(pd.DataFrame(geometry_rows))

,kind,pair,concept,reference,period,aliases,E,E_low,E_high,undefined_draw_fraction,...,delta_singleton,delta_low,delta_high,delta_undefined_fraction,delta_selected,delta_selected_low,delta_selected_high,delta_selected_undefined_fraction,energy_inside_A,energy_outside_A
0,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",0.543626,0.523090,0.570598,0.0,...,-0.566677,-0.587800,-0.539400,0.0,0.00000,0.000000,0.000000,0.0,NaN,NaN
1,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",0.452430,0.443069,0.464727,0.0,...,-0.659042,-0.668158,-0.646882,0.0,0.00000,0.000000,0.000000,0.0,NaN,NaN
2,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",0.589392,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.432393,5.079221
4,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",0.630133,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10075,control_3,135-215,"2014/42/42:73,135:222,215:96",2014,secondary,all_history,0.894110,0.842030,0.956213,0.0,...,0.096349,0.043681,0.157935,0.0,0.50529,0.454917,0.566684,0.0,NaN,NaN
10076,control_3,135-215,"2014/42/42:73,135:222,215:96",2014,secondary,all_history,0.486799,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10077,control_3,135-215,"2014/42/42:73,135:222,215:96",2014,secondary,all_history,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.035784,17.374131
10078,control_3,135-215,"2014/42/42:73,135:222,215:96",2014,secondary,all_history,0.478040,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,kind,pair,concept,reference,period,aliases,coordinate,angle,squared_cosine,left_rank,right_rank,left_projection_energy,right_projection_energy
0,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",raw,1,0.362950,5,5,0.301448,0.277827
1,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",raw,2,0.149360,5,5,0.301448,0.277827
2,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",raw,3,0.062134,5,5,0.301448,0.277827
3,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",raw,4,0.009335,5,5,0.301448,0.277827
4,selected,42-135,"2007/42/42:94,135:3,215:146",2007,primary,"last_3,all_history",raw,5,0.001883,5,5,0.301448,0.277827
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13579,control_3,135-215,"2014/42/42:73,135:222,215:96",2014,secondary,all_history,standardized,1,0.187602,5,5,0.170920,0.139440
13580,control_3,135-215,"2014/42/42:73,135:222,215:96",2014,secondary,all_history,standardized,2,0.054090,5,5,0.170920,0.139440
13581,control_3,135-215,"2014/42/42:73,135:222,215:96",2014,secondary,all_history,standardized,3,0.022631,5,5,0.170920,0.139440
13582,control_3,135-215,"2014/42/42:73,135:222,215:96",2014,secondary,all_history,standardized,4,0.007768,5,5,0.170920,0.139440


## Findings

Contribution agreement, geometry and anchor specificity answer different questions.
Show raw estimates, intervals, support and unavailable controls. No fixed threshold
labels a family reliable or equivalent. Overlapping families and shared early-year
aliases are descriptive observations, not independent replications.


In [14]:
if MODE != 'check':
    fields = ('concept', 'reference', 'pair', 'aliases', 'fit_rows', 'tune_rows', 'evaluation_rows', 'controls', 'seconds', 'status')
if MODE != "check":
    print(f"Support rows: {len(support_rows)}.")


Support rows: 336.


In [15]:
if MODE != 'check':
    publish(ctx, support_rows, fields)

In [16]:
if MODE != 'check' and bool(group_rows):
    fields = ('kind', 'pair', 'concept', 'reference', 'period', 'aliases', 'E', 'E_low', 'E_high', 'undefined_draw_fraction', 'left_rms', 'right_rms', 'zero_norm_fraction', 'mean_signed_cosine', 'stratum', 'rows', 'distinct_count', 'status', 'left_factors', 'right_factors', 'left_size', 'right_size', 'delta_singleton', 'delta_low', 'delta_high', 'delta_undefined_fraction', 'energy_inside_A', 'energy_outside_A', 'delta_selected', 'delta_selected_low', 'delta_selected_high', 'delta_selected_undefined_fraction')
    publish(ctx, group_rows, fields)
    publish(ctx, geometry_rows, ('kind', 'pair', 'concept', 'reference', 'period', 'aliases', 'coordinate', 'angle', 'squared_cosine', 'left_rank', 'right_rank', 'left_projection_energy', 'right_projection_energy'))
if MODE != "check" and group_rows:
    print(f"Report contribution records: {len(group_rows)}; frozen size-pair records: {len(curve_rows)}.")


Report contribution records: 10080; frozen size-pair records: 8400.


In [17]:
if MODE != 'check' and bool(group_rows):
    publish(ctx, curve_rows, ('concept', 'reference', 'pair', 'aliases', 'period', 'left_size', 'right_size', 'E', 'selected'))
    summary_rows, range_rows = ([], [])
    expanded = pd.DataFrame([{**row, 'variant': variant} for row in group_rows for variant in row['aliases'].split(',') if row.get('stratum') in ('full', 'original_high')])
if MODE != "check" and group_rows:
    print(f"Report contribution records: {len(group_rows)}; frozen size-pair records: {len(curve_rows)}.")


Report contribution records: 10080; frozen size-pair records: 8400.


In [18]:
if MODE != 'check' and bool(group_rows):
    for overall in (False, True):
        keys = ['variant', 'period', 'kind', 'stratum'] + ([] if overall else ['reference'])
        for identity, frame in expanded.groupby(keys):
            record = dict(zip(keys, identity))
            record.update(reference=None if overall else int(record['reference']), level='overall' if overall else 'reference', n_total=len(frame), n_computable=int(frame['E'].notna().sum()), median_E=float(frame['E'].median()) if frame['E'].notna().any() else None, median_delta_singleton=float(frame['delta_singleton'].median()) if frame['delta_singleton'].notna().any() else None, median_delta_selected=float(frame['delta_selected'].median()) if frame['delta_selected'].notna().any() else None)
            summary_rows.append(record)
if MODE != "check" and group_rows:
    print(f"Report contribution records: {len(group_rows)}; frozen size-pair records: {len(curve_rows)}.")


Report contribution records: 10080; frozen size-pair records: 8400.


In [19]:
if MODE != 'check' and bool(group_rows):
    control = expanded[expanded['kind'].str.startswith('control_')]
    for identity, frame in control.groupby(['variant', 'period', 'reference', 'concept', 'pair', 'stratum']):
        record = dict(zip(('variant', 'period', 'reference', 'concept', 'pair', 'stratum'), identity))
        record.update(n_expected=CONTROLS, n_computable=int(frame['E'].notna().sum()), control_min=float(frame['E'].min()) if frame['E'].notna().any() else None, control_max=float(frame['E'].max()) if frame['E'].notna().any() else None)
        range_rows.append(record)
if MODE != "check" and group_rows:
    print(f"Report contribution records: {len(group_rows)}; frozen size-pair records: {len(curve_rows)}.")


Report contribution records: 10080; frozen size-pair records: 8400.

In [20]:
if MODE != 'check' and bool(group_rows):
    display(publish(ctx, summary_rows, ('variant', 'period', 'reference', 'kind', 'stratum', 'level', 'n_total', 'n_computable', 'median_E', 'median_delta_singleton', 'median_delta_selected')))
    display(publish(ctx, range_rows, ('variant', 'period', 'reference', 'concept', 'pair', 'stratum', 'n_expected', 'n_computable', 'control_min', 'control_max')))
    curves = pd.DataFrame([{**row, 'variant': variant} for row in curve_rows for variant in row['aliases'].split(',')])
if MODE != "check" and group_rows:
    print(f"Report contribution records: {len(group_rows)}; frozen size-pair records: {len(curve_rows)}.")


,kind,level,median_E,median_delta_selected,median_delta_singleton,n_computable,n_total,period,reference,stratum,variant
0,control_1,reference,0.566668,0.078530,-0.318907,12,12,primary,2007.0,full,all_history
1,control_1,reference,0.519261,0.040870,-0.328214,18,18,primary,2008.0,full,all_history
2,control_1,reference,0.571963,0.143002,-0.326850,30,30,primary,2009.0,full,all_history
3,control_1,reference,0.558354,0.095423,-0.303933,24,24,primary,2010.0,full,all_history
4,control_1,reference,0.592484,-0.028464,-0.355787,15,15,primary,2011.0,full,all_history
...,...,...,...,...,...,...,...,...,...,...,...
195,control_3,overall,0.639187,0.265122,-0.255575,24,24,secondary,NaN,original_high,last_3
196,selected,overall,0.456691,0.000000,-0.425342,24,24,secondary,NaN,full,last_3
197,selected,overall,0.405373,0.000000,-0.481886,24,24,secondary,NaN,original_high,last_3
198,singleton,overall,0.868555,0.425342,0.000000,24,24,secondary,NaN,full,last_3


,concept,control_max,control_min,n_computable,n_expected,pair,period,reference,stratum,variant
0,"2007/42/42:1,135:175,215:120",0.603875,0.472374,3,3,135-215,primary,2007,full,all_history
1,"2007/42/42:1,135:175,215:120",0.910357,0.456084,3,3,135-215,primary,2007,original_high,all_history
2,"2007/42/42:1,135:175,215:120",0.659617,0.453524,3,3,42-135,primary,2007,full,all_history
3,"2007/42/42:1,135:175,215:120",0.643302,0.428689,3,3,42-135,primary,2007,original_high,all_history
4,"2007/42/42:1,135:175,215:120",0.635295,0.543531,3,3,42-215,primary,2007,full,all_history
...,...,...,...,...,...,...,...,...,...,...
787,"2014/42/42:96,135:32,215:116",0.637461,0.420703,3,3,135-215,secondary,2014,original_high,last_3
788,"2014/42/42:96,135:32,215:116",0.640997,0.459061,3,3,42-135,secondary,2014,full,last_3
789,"2014/42/42:96,135:32,215:116",0.643984,0.466814,3,3,42-135,secondary,2014,original_high,last_3
790,"2014/42/42:96,135:32,215:116",0.620005,0.528196,3,3,42-215,secondary,2014,full,last_3


Report contribution records: 10080; frozen size-pair records: 8400.


In [21]:
if MODE != 'check' and bool(group_rows):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8), squeeze=False)
    for ax, (variant, period) in zip(axes.flat, itertools.product(('last_3', 'all_history'), ('primary', 'secondary'))):
        scoped = curves[(curves['variant'] == variant) & (curves['period'] == period)]
        plot = scoped.groupby(['left_size', 'right_size'])['E'].median().unstack()
        if plot.empty:
            ax.set(title=f'{variant}: {period}; no computable curves')
            continue
        image = ax.imshow(plot.to_numpy(), origin='lower', vmin=0, aspect='auto')
        ax.set(xticks=range(len(plot.columns)), xticklabels=plot.columns, yticks=range(len(plot.index)), yticklabels=plot.index, xlabel='Right group size', ylabel='Left group size', title=f"{variant}: {period}; {scoped[['concept', 'reference', 'pair']].drop_duplicates().shape[0]} pairs")
        fig.colorbar(image, ax=ax, label='Median frozen E')
if MODE != "check" and group_rows:
    print(f"Report contribution records: {len(group_rows)}; frozen size-pair records: {len(curve_rows)}.")


Report contribution records: 10080; frozen size-pair records: 8400.


In [22]:
if MODE != 'check' and bool(group_rows):
    fig.tight_layout()
    plt.show()
    print('Shared aliases shown in both variant summaries; intervals never pool them as independent replications.')
if MODE != "check" and group_rows:
    print(f"Report contribution records: {len(group_rows)}; frozen size-pair records: {len(curve_rows)}.")


Shared aliases shown in both variant summaries; intervals never pool them as independent replications.
Report contribution records: 10080; frozen size-pair records: 8400.


In [23]:
if MODE != 'check':
    print(f'Distinct family/seed-pair units: {len(support_rows)}; no mortality outcomes read.')
    gaps = [item for item in ctx['coverage'] if item['status'] != 'measured']
    display(publish(ctx, gaps, ('build', 'reference', 'status', 'n')))

Distinct family/seed-pair units: 336; no mortality outcomes read.


,build,n,reference,status
0,last_3,1,2015,missing_dependency
1,all_history,1,2015,missing_dependency
2,last_3,1,2015,missing_dependency
3,all_history,1,2015,missing_dependency


## Checks

Synthetic fixtures call the same helpers used above: exact moments, split-signal
recovery, tuning independence, bounded controls, aliases, ranks and coordinates.
Check mode performs no real input loading, fitting or exports.


In [24]:
CHECK_ROWS = []
results = CHECK_ROWS
orthogonal = (np.eye(2), np.eye(2), np.zeros((2, 2)))
assert np.isclose(moment_error(orthogonal, (0,), (0,)), 1.), "orthogonal contributions must have E=1"
results.append({"check": "normalized disagreement", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,normalized disagreement,passed


In [25]:
rng = np.random.default_rng(42)
a, b = rng.uniform(0, 1, (30, 7)), rng.uniform(0, 1, (30, 8))
d, e = rng.normal(size=(7, 4)), rng.normal(size=(8, 4))
moments = contribution_moments(a, b, d, e, "cpu")
left, right = (0, 2), (1, 3, 5)
c, f = actual_contribution(a, d, left), actual_contribution(b, e, right)
n, den = row_energies(c, f)
assert np.isclose(moment_error(moments, left, right), n.sum()/den.sum())
assert moment_error(tuple(np.zeros_like(m) for m in moments), left, right) is None
results.append({"check": "moment/direct equality and zero energy", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,moment/direct equality and zero energy,passed


In [26]:
x = rng.uniform(.1, 1, (40, 1))
split_a, split_b = np.column_stack([x[:, 0], x[:, 0]]), x.copy()
split_d, split_e = np.array([[1., 0.], [0., 1.]]), np.array([[1., 1.]])
split_m = contribution_moments(split_a, split_b, split_d, split_e, "cpu")
searched = bounded_search(split_m, (0, 0), (alive_factors(split_a), alive_factors(split_b)), ({0}, {0}))
tuned = tune_search(searched, split_m)
assert tuned["selected"] == ((0, 1), (0,))
assert moment_error(split_m, *tuned["selected"]) < moment_error(split_m, (0,), (0,))
results.append({"check": "planted split-signal recovery", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,planted split-signal recovery,passed


In [27]:
search = bounded_search(moments, (0, 1), (alive_factors(a), alive_factors(b)), ({0}, {1}))
assert len(search["candidates"]) == 25 and max(map(len, search["candidates"].values())) <= BEAM
assert all(len(s) <= SHORTLIST for s in search["shortlists"])
assert search == bounded_search(moments, (0, 1), (alive_factors(a), alive_factors(b)), ({0}, {1}))
assert all(0 in pair[0] and 1 in pair[1] for v in search["candidates"].values() for pair in v)
assert ordered_candidates([((0,), (0,1)), ((0,), (0,))], lambda pair: 0., tuning=True)[0] == ((0,), (0,))
results.append({"check": "deterministic bounds, tuning ties and anchor retention", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,"deterministic bounds, tuning ties and anchor r...",passed


In [28]:
controls = matched_anchors((a, b), (d, e), (0, 1), ({0}, {1}))
assert len(controls) == 3 and len(set(p[0] for p in controls)) == 3 and len(set(p[1] for p in controls)) == 3
assert all(p[0] != 0 and p[1] != 1 for p in controls)
for pair in controls:
    control_search = bounded_search(moments, pair, (alive_factors(a), alive_factors(b)), ({0}, {1}))
    assert len(control_search["candidates"]) == len(search["candidates"])
    assert all(0 not in p[0] and 1 not in p[1] for v in control_search["candidates"].values() for p in v)
assert matched_anchors((a[:,:1], b[:,:1]), (d[:1], e[:1]), (0,0), ({0},{0})) == []
results.append({"check": "equal control capacity and unavailable matches", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,equal control capacity and unavailable matches,passed


In [29]:
original = tune_search(search, moments)
poisoned_evaluation = np.full((30, 4), 1e30)
assert original == tune_search(search, moments) and poisoned_evaluation.max() == 1e30
results.append({"check": "development takes fit/tune moments only", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,development takes fit/tune moments only,passed


In [30]:
assert PAIRS == ((42, 135), (42, 215), (135, 215))
g = group_geometry(np.array([[1., 0.], [2., 0.]]), np.eye(2), np.array([[3., 0.]]), np.array([[1., 1.]]))
assert g["left_rank"] == 1 and g["right_rank"] == 2 and np.allclose(g["spectrum"], 1)
assert np.isclose(g["left_projection_energy"], 1.) and np.isclose(g["right_projection_energy"], .5)
results.append({"check": "three pairs and directed unequal-rank geometry", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,three pairs and directed unequal-rank geometry,passed


In [31]:
weights = np.ones((3, 30))
diag = contribution_diagnostics(c, f, weights)
assert np.isclose(diag["E"], n.sum()/den.sum()) and diag["undefined_draw_fraction"] == 0
empty = contribution_diagnostics(c[:0], f[:0], weights[:, :0])
assert empty["E"] is None and empty["undefined_draw_fraction"] == 1
results.append({"check": "conditional weighted errors and empty strata", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,conditional weighted errors and empty strata,passed


In [32]:
unit = {"fit": np.arange(20), "tune": np.arange(20, 30), "codes": {42:a,135:b,215:b},
        "decoders":{42:d,135:e,215:e}, "scale":np.array([1.,2.,3.,4.]), "members":{42:0,135:1,215:2},
        "identity":{"frozen":"synthetic"}, "labels":np.zeros(30), "aliases":["last_3"]}
developed = develop_pair(unit, (135,215), "cpu")
assert developed["true"]["anchors"] == (1,2)
unit["labels"][:] = 1e30
unit["codes"] = {seed: np.vstack([values, np.full((5, values.shape[1]), 1e30)])
                 for seed, values in unit["codes"].items()}
unit["reference"] = 2015
unit["aliases"] = ["last_3", "all_history"]
assert developed == develop_pair(unit, (135,215), "cpu")
results.append({"check": "non-medoid anchors, alias reuse, outcome poisoning", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,"non-medoid anchors, alias reuse, outcome poiso...",passed


In [33]:
scale = np.array([2., .5, 3., 1.])
scaled = contribution_moments(a, b, d * scale, e * scale, "cpu")
sc, sf = c * scale, f * scale
sn, sd = row_energies(sc, sf)
assert np.isclose(moment_error(scaled, left, right), sn.sum()/sd.sum())
results.append({"check": "raw coordinate scaling", "result": "passed"})
display(pd.DataFrame([results[-1]]))

,check,result
0,raw coordinate scaling,passed


In [34]:
CHECK_RESULTS = pd.DataFrame(CHECK_ROWS)
display(CHECK_RESULTS)
if MODE != "check":
    finish_run(ctx, "completed", {"completed_pairs": len(support_rows), "checks": len(CHECK_RESULTS)})

,check,result
0,normalized disagreement,passed
1,moment/direct equality and zero energy,passed
2,planted split-signal recovery,passed
3,"deterministic bounds, tuning ties and anchor r...",passed
4,equal control capacity and unavailable matches,passed
5,development takes fit/tune moments only,passed
6,three pairs and directed unequal-rank geometry,passed
7,conditional weighted errors and empty strata,passed
8,"non-medoid anchors, alias reuse, outcome poiso...",passed
9,raw coordinate scaling,passed
